# Train — GRPO on Colab T4

Fine-tune Qwen3-0.6B on your OpenEnv environment using GRPO via Unsloth 4-bit LoRA.

**Runtime:** Colab T4 (free tier). ~8–15 minutes for a short demo run; hours for production.

**What happens:**
1. Install training deps (TRL + Unsloth + torch)
2. Build a prompt dataset from env resets
3. GRPOTrainer samples N completions per prompt → each completion is an action → env returns reward
4. Save LoRA adapter + push to Hugging Face

## Check GPU

In [ ]:
!nvidia-smi | head -20

## Install the template + training deps

⚠️ T4 gotchas baked into `grpo_unsloth.py`:
- BF16 LoRA (not FP8 — T4 doesn't support FP8)
- `use_vllm=False` (conflicts with Unsloth GRPO)
- `use_gradient_checkpointing='unsloth'` (memory)

In [ ]:
import os
REPO = 'openenv-r2-kit'
if not os.path.exists(REPO):
    !git clone -q https://github.com/Kaviyamurugadass/openenv-r2-kit.git
%cd {REPO}

!pip install -q uv
# Colab-compatible install: Unsloth bundles compatible torch/trl wheels
!pip install -q unsloth unsloth_zoo
!pip install -q 'trl>=0.29' 'transformers>=5.3' datasets accelerate
!uv sync --quiet

## HF token (for pushing the trained adapter later)

In [ ]:
from huggingface_hub import login
from google.colab import userdata  # Colab secret
try:
    login(token=userdata.get('HF_TOKEN'))
except Exception:
    # Fallback: interactive login if no Colab secret set
    login()

## 1. Collect baseline rollouts (fills the 'Baseline' column)

In [ ]:
import sys; sys.path.insert(0, '.')
from training.rollout_collection import collect

for policy in ['random', 'heuristic']:
    ds = collect(
        episodes=10,
        policy=policy,
        output_dir=f'data/baseline/{policy}',
        seed=42,
        domain_randomise=True,
    )
    print(f'{policy}: {ds.summary()}')

## 2. Dry-run the GRPO pipeline (verify setup)

In [ ]:
!python training/grpo_unsloth.py --dry-run

## 3. Run real training — T4-optimized flags

Short demo run (8 episodes, 1 epoch, batch=1). For production, scale up `--dataset-episodes` and `--num-epochs`.

In [ ]:
!python training/grpo_unsloth.py \
    --model-id Qwen/Qwen3-0.6B \
    --output-dir training/grpo-unsloth-qwen3-0.6b \
    --dataset-episodes 8 \
    --rollout-steps 6 \
    --per-device-train-batch-size 1 \
    --num-generations 2 \
    --gradient-accumulation-steps 4 \
    --max-seq-length 1024 \
    --lora-r 16 \
    --learning-rate 5e-6 \
    --num-epochs 1 \
    --trust-remote-code

## 4. Quick post-training evaluation

In [ ]:
# Re-import to pick up any changes
from training.rollout_collection import collect
from training.evaluation import EvaluationSuite

# NOTE: this uses the template's heuristic policy as a proxy for the trained
# agent. Replace with an LLM-backed policy that loads your trained adapter
# (see training/grpo_unsloth.py inference loop or inference.py) to measure
# the actual trained behavior.
trained_ds = collect(
    episodes=10,
    policy='heuristic',
    output_dir='data/trained',
    seed=42,
)

print('Trained:', EvaluationSuite.online_metrics(trained_ds))

## 5. Push trained adapter to Hugging Face

Creates a public model repo with your LoRA + tokenizer. Link it from the env README.

In [ ]:
# Change USER to your HF namespace
USER = 'Kaviya-M'
MODEL_NAME = 'openenv-r2-kit-grpo-qwen3-0.6b'

!hf upload {USER}/{MODEL_NAME} training/grpo-unsloth-qwen3-0.6b

## Next steps

- **Evaluate properly** (baseline vs trained, fill README table): `03_evaluate.ipynb`
- **Tune hyperparameters:** see `training/grpo_unsloth.py --help` for the full flag list
- **Scale up:** `--dataset-episodes 32 --num-epochs 3 --num-generations 4` on A100